# 🏗️ Notebook 1: Discord — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

Real-time chat for large **guilds** (servers) with text channels, voice channels, presence,
and push notifications.

### Functional requirements
- Send/receive **text messages** in channels within <200ms.
- Users see **online/offline presence** of friends and guild members.
- **Voice channels** (low-latency audio).
- Message history is preserved (scrollback).
- Push notifications when user is offline.

### Non-functional
- **Real-time**: WebSockets for delivery, not HTTP polling.
- **Massive fan-out**: a guild channel can have 100k+ members; a single message must reach them all.
- **Geographically distributed** voice servers for low latency.


## Back-of-envelope

- 150M MAU, ~15M concurrent. ~2 messages/user/min → **500k msgs/sec** peak.
- Each message fanned out to an average of 50 recipients → **25M events/sec delivered**.
- Voice: a ~64 kbps stream per user; 1M concurrent voice users = 64 Gbps.


## High-level architecture

```
   [client]
     │ wss://gateway...
     ▼
 ┌──────────────┐        ┌──────────────┐
 │  Gateway     │◀──────▶│ Session/     │  (who is online, which shard)
 │  (WebSocket) │        │ Presence     │
 └──────┬───────┘        └──────────────┘
        │ publishes
        ▼
 ┌──────────────┐        ┌─────────────┐
 │ Message Bus  │───────▶│ Channel     │
 │ (Kafka/NATS) │        │ Fan-out Svc │
 └──────────────┘        └─────────────┘
        │                      │
        ▼                      ▼
 ┌──────────────┐        ┌─────────────┐
 │ Msg Storage  │        │ Push Notif  │
 │ (Cassandra)  │        │ (offline)   │
 └──────────────┘        └─────────────┘

  Voice: separate UDP media servers, WebRTC.
```

### Gateway sharding
A single WebSocket server can handle ~50k connections. With 15M concurrent users:
**15M / 50k = 300 gateway instances**. Clients are assigned via consistent hashing of user_id.
